In [ ]:
%pip install -q langchain langchain-openai langchain-community chromadb langchainhub

In [ ]:
%pip install -q python-dotenv
from dotenv import load_dotenv
load_dotenv()

In [ ]:
#################################################################
# Select an embeddings model
#################################################################
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

In [ ]:

#  pip install chromadb
# https://python.langchain.com/docs/integrations/vectorstores/chroma#passing-a-chroma-client-into-langchain

import chromadb

chroma_client = chromadb.PersistentClient(path="./chromadb")
from langchain_community.vectorstores import Chroma

vectorstore = Chroma(
    # persist_directory="./chromadb",  # by default it is memory only
    collection_name="my_langchain",  # the default chromadb collection is langchain
    client=chroma_client,
    embedding_function=embeddings,  # different from  Chroma.from_documents
    # https://github.com/langchain-ai/langchain/blob/master/libs/community/langchain_community/vectorstores/chroma.py
    # client_settings
)

# Retrieve and generate using the relevant snippets of the blog.
retriever = vectorstore.as_retriever()
# docs = retriever.get_relevant_documents("What is devopsdays?")
# print(docs)

In [ ]:

#################################################################
# Get a predefined prompt for RAG
# from the langchain hub
# https://smith.langchain.com/hub
#################################################################

# needs pip install langchainhub
from langchain import hub

prompt = hub.pull("rlm/rag-prompt")

Now let's inspect the prompt we got.
> Classic track note: This notebook demonstrates legacy/classic LangChain-era patterns for evaluation and comparison.
> Prefer the modern equivalents in `lessons/2026-langchain/` for current APIs and recommended techniques.


In [ ]:
from pprint import pprint
pprint(prompt.input_variables)
a=prompt.format_prompt(context="some context",question="some question")
for msg in a.messages:
    print(msg.content)

In [ ]:
# Configure a Chat LLM
from langchain_openai import ChatOpenAI

chat = ChatOpenAI(temperature=0)

In [ ]:

# helper function to join a set of documents
# Retrieved from the vectorstore
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:

#################################################################
# Setup our rag chain
# Using LCEL : Langchain  Expression Language
# https://python.langchain.com/docs/expression_language/
#################################################################
from langchain_core.runnables import RunnablePassthrough

from langchain_core.output_parsers.string import StrOutputParser
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | chat
    | StrOutputParser()
)

answer = rag_chain.invoke("What is Devopsdays ?")
print(answer)
